# Sesión 4 — Sistemas de Recomendación

Código de los conceptos de la sesión y el ejercicio para practicarlos.

## 1. ¿Qué es un sistema de recomendación?

Un **sistema de recomendación** es un enfoque basado en software que sugiere
ítems personalizados que probablemente sean de interés para un usuario
particular, a partir de datos históricos de interacción (y, opcionalmente,
de atributos del usuario y de los ítems).

Están en todas partes: el "Top 10 para ti" de Netflix, el feed de
recomendados de YouTube o TikTok, "productos que te podrían interesar" de
Amazon o Mercado Libre, "personas que quizás conozcas" de LinkedIn, las
playlists de descubrimiento de Spotify. En muchas de estas plataformas, el
recomendador **es** el producto: gran parte del tiempo de uso y de los
ingresos se explican por qué tan bien el sistema elige qué mostrar.

La pregunta central que resuelve un recomendador es: **de todo el catálogo,
¿qué le muestro primero a este usuario, en este momento?**

## 2. Formulación del problema

Hay dos maneras (relacionadas, pero distintas) de formular matemáticamente
el problema de recomendación:

### (a) Predicción de rating (regresión)

Estimar $\hat{r}(u, i)$: el valor numérico que el usuario $u$ le daría al
ítem $i$ si lo calificara. Se entrena **minimizando un error de regresión**
(RMSE o MAE) entre el rating predicho y el rating real observado en un
conjunto de validación. Es la formulación que usó el **Netflix Prize**
(2006-2009): predecir, en una escala de 1 a 5 estrellas, qué rating le
daría un usuario a una película que no ha visto.

### (b) Tarea de ranking

Dado un usuario, **ordenar** los ítems del catálogo por relevancia y
mostrarle los primeros $K$. Aquí no importa tanto acertar el valor exacto
del rating, sino el **orden relativo**: si el usuario va a interactuar con
los primeros 10 ítems que le mostremos, lo que importa es que esos 10 sean
los más relevantes posibles, no que prediagamos "3.8" en vez de "4.1"
para uno de ellos.

En la práctica, la mayoría de sistemas modernos (recomendación de feed,
e-commerce, streaming) se preocupan sobre todo por el **ranking** — el
usuario casi nunca ve un rating predicho explícito, solo una lista
ordenada. Pero entrenar un buen predictor de rating (o de una señal de
afinidad continua) suele ser el primer paso para poder ordenar.

## 3. Tipos de feedback: explícito vs. implícito

| | **Feedback explícito** | **Feedback implícito** |
|---|---|---|
| Qué es | Calificación deliberada del usuario | Comportamiento observado |
| Ejemplos | Estrellas, me gusta / no me gusta, reseñas | Clicks, compras, tiempo de visualización, scroll, skip, "agregar al carrito" |
| Señal | Limpia, sabemos exactamente qué opina el usuario | Ruidosa: un click no siempre significa interés real (curiosidad, error, clickbait) |
| Volumen | Escasa: pocos usuarios se toman el trabajo de calificar | Abundante: se genera todo el tiempo, sin esfuerzo del usuario |
| Sesgo | Sesgo de autoselección (solo calificamos lo que amamos u odiamos) | Sesgo de exposición (solo observamos interacción con lo que el sistema ya mostró) |

En la práctica, la gran mayoría de sistemas de recomendación de producción
(Netflix, YouTube, TikTok, e-commerce) dependen mucho más de **feedback
implícito** que de feedback explícito, precisamente porque hay órdenes de
magnitud más señal disponible. MovieLens, el dataset que usaremos hoy, es
una rara excepción con feedback **explícito** limpio (ratings de 1 a 5
estrellas dados deliberadamente) — lo cual lo hace ideal para enseñar la
formulación de predicción de rating, aunque no sea representativo del tipo
de dato que vas a encontrar en la mayoría de sistemas reales.

## 4. Pipeline de un recomendador moderno a gran escala

Los sistemas de recomendación de producción (piensa en YouTube o TikTok,
con cientos de millones de ítems y usuarios) **no pueden** puntuar todo el
catálogo con un modelo pesado para cada usuario en tiempo real — sería
computacionalmente inviable. Por eso usan un pipeline en **etapas**, cada
una reduciendo el número de candidatos y aumentando la sofisticación (y el
costo computacional) del modelo:

```
 Catálogo completo           Retrieval              Ranking             Re-Ranking
 (10^6 - 10^9 ítems)     (~10^2 - 10^3 candidatos)  (~10^2 candidatos)   (Top-N final)
┌──────────────────┐      ┌──────────────────┐    ┌──────────────────┐  ┌──────────────────┐
│  Todo el          │ ---> │ Modelos rápidos,  │--->│ Modelo pesado,   │->│ Ajusta la lista   │
│  catálogo         │      │ alto recall       │    │ features ricos   │  │ final: diversidad,│
│  disponible       │      │ (embeddings+kNN   │    │ de contexto      │  │ novedad, fairness  │
│                   │      │ aproximado,       │    │ (Gradient        │  │ (no solo mostrar   │
│                   │      │ two-tower,        │    │ Boosting,        │  │ 10 ítems del       │
│                   │      │ FAISS/HNSW)       │    │ Wide & Deep)     │  │ mismo tipo)        │
└──────────────────┘      └──────────────────┘    └──────────────────┘  └──────────────────┘
```

- **Retrieval (candidate generation):** el objetivo es *recall*, no
  precisión — de millones de ítems, encontrar rápidamente unos cientos que
  con alta probabilidad contengan los relevantes. Se usan modelos livianos:
  embeddings de usuario e ítem (ej. arquitecturas **two-tower**) buscados
  con estructuras de vecinos aproximados (**ANN**, ej. FAISS, HNSW), o
  simplemente filtros de negocio (popularidad, categoría).
- **Ranking:** sobre esos ~100-1000 candidatos, un modelo mucho más pesado
  (que sí puede usar cientos de features de contexto: hora del día,
  dispositivo, historial reciente, señales de calidad del ítem) los
  reordena con precisión. Ejemplos típicos: Gradient Boosting, Wide & Deep,
  redes neuronales profundas.
- **Re-Ranking:** la lista ya está ordenada por relevancia pura, pero
  mostrar los 10 ítems "más relevantes" a secas puede ser mala idea — por
  ejemplo, 10 variaciones del mismo producto, o contenido que nunca da
  visibilidad a productores pequeños. Esta última etapa ajusta el orden
  final para meter diversidad, ítems novedosos, o restricciones de
  fairness/negocio.

Hoy en el notebook vamos a trabajar el problema "sin las etapas de
escala" — MovieLens 100k tiene solo 1682 películas, así que podemos
puntuar *todo* el catálogo con nuestro modelo de factorización directamente
(no necesitamos retrieval). Pero la lógica de fondo (predecir afinidad,
rankear, y luego pensar más allá del ranking puro) es exactamente la
misma.

## 5. La matriz de utilidad

El objeto central de un sistema de recomendación es la **matriz de
utilidad** $R$, de $m$ usuarios × $n$ ítems, donde $r_{ui}$ es la afinidad
observada del usuario $u$ por el ítem $i$ (un rating, un click, un tiempo
de visualización, etc).

La inmensa mayoría de las celdas de $R$ están **vacías**: ningún usuario
interactúa con más que una fracción minúscula del catálogo. La tarea
fundamental de un sistema de recomendación es, en el fondo, **completar las
celdas faltantes de esta matriz** (o al menos, estimar cuáles serían las
más altas) — ya sea con content-based filtering, collaborative filtering,
o alguna combinación.

Vamos a construir esta matriz con MovieLens 100k y a medir qué tan vacía
está en la práctica.

## Preparación del entorno

Descargamos MovieLens 100k en tiempo de ejecución (no se comitea al repo:
son datos públicos, se vuelven a descargar cada vez que se corre el
notebook).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_DIR = Path("ml100k_data")
RAW_DIR = DATA_DIR / "ml-100k"

pd.set_option("display.max_columns", 20)

In [ ]:
# Datos alojados en el almacenamiento del curso.
CDN = "https://d3qixogk4zgixq.cloudfront.net/data/sesiones/sesion_06_ml100k/ml-100k"
ARCHIVOS = ("u.data", "u.item", "u.user", "u.genre", "u.info", "u.occupation")

RAW_DIR.mkdir(parents=True, exist_ok=True)
for archivo in ARCHIVOS:
    destino = RAW_DIR / archivo
    if not destino.exists():
        r = requests.get(f"{CDN}/{archivo}", timeout=60)
        r.raise_for_status()
        destino.write_bytes(r.content)
print("Datos listos en", RAW_DIR)

### Carga de interacciones (`u.data`) y metadata de películas (`u.item`)

`u.data` trae tuplas `(user_id, item_id, rating, timestamp)` separadas por
tabulador. `u.item` trae, separado por `|`: id, título, fecha de estreno,
URL de IMDb, y 19 columnas binarias de género (una película puede tener
más de un género activo).

In [ ]:
ratings = pd.read_csv(
    RAW_DIR / "u.data",
    sep="\t",
    names=["user_id", "item_id", "rating", "timestamp"],
    engine="python",
)

GENRE_COLS = [
    "unknown", "Action", "Adventure", "Animation", "Children's", "Comedy",
    "Crime", "Documentary", "Drama", "Fantasy", "Film-Noir", "Horror",
    "Musical", "Mystery", "Romance", "Sci-Fi", "Thriller", "War", "Western",
]
ITEM_COLS = ["item_id", "title", "release_date", "video_release_date", "imdb_url"] + GENRE_COLS

movies = pd.read_csv(
    RAW_DIR / "u.item",
    sep="|",
    names=ITEM_COLS,
    encoding="latin-1",
    engine="python",
)[["item_id", "title"] + GENRE_COLS]

print(ratings.shape, movies.shape)
ratings.head()

In [ ]:
movies.head()

### Construcción de la matriz de utilidad y medición de la sparsity real

In [ ]:
n_usuarios = ratings["user_id"].nunique()
n_peliculas = ratings["item_id"].nunique()

R = ratings.pivot(index="user_id", columns="item_id", values="rating")
print(f"Matriz de utilidad R: {R.shape[0]} usuarios x {R.shape[1]} películas")

n_celdas = R.shape[0] * R.shape[1]
n_observadas = ratings.shape[0]
density = n_observadas / n_celdas
sparsity = 1 - density

print(f"Celdas totales: {n_celdas:,}")
print(f"Celdas observadas (ratings): {n_observadas:,}")
print(f"Densidad: {density:.4%}  |  Sparsity: {sparsity:.4%}")
R.iloc[:8, :8]

Con MovieLens 100k, la densidad real que medimos es de apenas un poco más
del **6%** — es decir, más del **93% de las celdas de la matriz de utilidad
están vacías**. Y este dataset es, comparativamente, uno de los "densos":

- **MovieLens 100k** (el que usamos hoy): ~6.3% de densidad.
- **MovieLens 1M** (misma familia, más usuarios/películas): ~4.3% de
  densidad.
- **Netflix Prize** (100M de ratings, 480k usuarios, 17k películas): ~1.2%
  de densidad.

La tendencia es clara: entre más grande y realista es el catálogo, **más
sparse** es la matriz de utilidad. Esto no es un detalle incidental, es
*la* dificultad estructural del problema.

## 6. Sparsity y Cold-Start

La altísima sparsity de la matriz de utilidad tiene dos implicaciones
directas:

1. **Alta varianza por usuario/ítem.** Cada usuario solo calificó un
   puñado de películas de las 1682 disponibles; cada película fue
   calificada solo por un puñado de los 943 usuarios. Cualquier estimación
   basada en pocos pares $(u,i)$ es estadísticamente ruidosa — por eso la
   regularización (que veremos en la factorización de matrices) es
   crítica, no opcional.
2. **Cold-start.** ¿Qué le recomendamos a un usuario que **acaba de
   registrarse** y no tiene ninguna interacción todavía? ¿Qué hacemos con
   una película **recién agregada** al catálogo, que nadie ha calificado
   aún? Ninguna técnica que dependa *solo* de patrones colectivos de
   interacción (collaborative filtering puro) puede decir nada útil en
   estos casos — no hay señal de la que aprender. Es uno de los
   principales motivadores para combinar collaborative filtering con
   content-based filtering (que sí puede usar atributos del ítem/usuario
   incluso sin historial de interacción).

In [ ]:
interacciones_por_usuario = ratings.groupby("user_id").size()
interacciones_por_pelicula = ratings.groupby("item_id").size()

print("Interacciones por usuario -> min:", interacciones_por_usuario.min(),
      "mediana:", interacciones_por_usuario.median(),
      "max:", interacciones_por_usuario.max())
print("Interacciones por película -> min:", interacciones_por_pelicula.min(),
      "mediana:", interacciones_por_pelicula.median(),
      "max:", interacciones_por_pelicula.max())

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(interacciones_por_usuario, bins=40, color="#4C72B0")
axes[0].set_title("Interacciones por usuario")
axes[0].set_xlabel("# ratings dados")
axes[1].hist(interacciones_por_pelicula, bins=40, color="#DD8452")
axes[1].set_title("Interacciones por película")
axes[1].set_xlabel("# ratings recibidos")
plt.tight_layout()
plt.show()

La distribución es muy asimétrica en ambos casos (cola larga): unos pocos
usuarios "power raters" y unas pocas películas "blockbuster" concentran
buena parte de las interacciones, mientras que muchos usuarios y muchas
películas tienen pocas — esos son, precisamente, los casos con mayor
varianza y el terreno donde el cold-start pega más duro.

## 7. Content-Based Filtering

La idea del content-based filtering es: **"te recomiendo cosas parecidas a
lo que te ha gustado"**.

- Cada ítem se representa como un **vector de features** (en nuestro caso,
  géneros binarios; en otros dominios podrían ser tags, embeddings de
  texto de una descripción, embeddings de imagen de una portada, etc).
- El **perfil del usuario** se construye a partir de los ítems con los que
  ha interactuado (ej. el promedio, ponderado por rating, de los vectores
  de las películas que calificó).
- Para recomendar, se calcula la **similitud coseno** entre el perfil del
  usuario y cada ítem del catálogo (o entre ítems, para "más como este"),
  y se recomiendan los más similares que el usuario no ha visto.

**Ventajas:**

- Funciona **desde el día 1** — no necesita el historial de *otros*
  usuarios, solo atributos del ítem. Por eso maneja muy bien el
  **cold-start de ítems** (una película nueva ya tiene género desde que se
  agrega al catálogo).
- Las recomendaciones son **explicables**: "te recomendamos *Toy Story 2*
  porque te gustó *Toy Story*, y ambas son Animación / Comedia / Infantil".

**Desventajas:** solo puede recomendar ítems parecidos a lo que ya te ha
gustado (poca serendipia), y depende de que existan features de calidad
para los ítems.

In [ ]:
item_features = movies.set_index("item_id")[GENRE_COLS]
print("Matriz de features de ítems:", item_features.shape)
item_features.head()

In [ ]:
# Similitud coseno ítem-ítem, a partir de los vectores de género
item_similarity = cosine_similarity(item_features.values)
item_similarity_df = pd.DataFrame(
    item_similarity, index=item_features.index, columns=item_features.index
)

# Ejemplo: películas más "parecidas" (por género) a Toy Story (item_id=1)
toy_story_id = movies.loc[movies["title"].str.contains("Toy Story"), "item_id"].iloc[0]
mas_similares = item_similarity_df[toy_story_id].sort_values(ascending=False)[1:8]
movies.set_index("item_id").loc[mas_similares.index, ["title"]].assign(similitud=mas_similares.values)

In [ ]:
def perfil_usuario_content_based(user_id: int, ratings_df: pd.DataFrame, item_features: pd.DataFrame) -> np.ndarray:
    '''Construye el perfil de un usuario como el promedio de los vectores de
    género de las películas que calificó, ponderado por su rating.'''
    vistas = ratings_df.loc[ratings_df["user_id"] == user_id, ["item_id", "rating"]]
    vectores = item_features.loc[vistas["item_id"]].to_numpy()
    pesos = vistas["rating"].to_numpy()
    perfil = (vectores * pesos[:, None]).sum(axis=0) / pesos.sum()
    return perfil


def recomendar_content_based(user_id: int, ratings_df: pd.DataFrame, item_features: pd.DataFrame,
                              movies_df: pd.DataFrame, top_n: int = 10) -> pd.DataFrame:
    '''Recomienda las top_n películas NO vistas más similares al perfil del usuario.'''
    perfil = perfil_usuario_content_based(user_id, ratings_df, item_features)
    similitudes = cosine_similarity(perfil.reshape(1, -1), item_features.to_numpy())[0]
    scores = pd.Series(similitudes, index=item_features.index, name="score_contenido")

    vistas_ids = set(ratings_df.loc[ratings_df["user_id"] == user_id, "item_id"])
    scores = scores.drop(index=vistas_ids, errors="ignore")

    top = scores.sort_values(ascending=False).head(top_n)
    return movies_df.set_index("item_id").loc[top.index, ["title"]].assign(score_contenido=top.values)


# Demo para un usuario de ejemplo
usuario_demo = 1
print(f"Películas mejor calificadas por el usuario {usuario_demo}:")
display(
    ratings.loc[ratings["user_id"] == usuario_demo]
    .merge(movies[["item_id", "title"]], on="item_id")
    .sort_values("rating", ascending=False)
    .head(5)[["title", "rating"]]
)

print(f"\nRecomendaciones content-based para el usuario {usuario_demo}:")
recomendar_content_based(usuario_demo, ratings, item_features, movies, top_n=10)

## 8. Collaborative Filtering

La idea del collaborative filtering es distinta: **"personas como tú
también vieron / calificaron bien..."**.

No usa atributos de ítem ni de usuario — solo la **matriz de interacciones**
$R$. Aprende patrones colectivos: si muchos usuarios que calificaron alto
las películas A y B también calificaron alto la película C, el sistema
puede sugerir C a un usuario nuevo que calificó alto A y B, **sin saber
nada** sobre el contenido de ninguna de las tres.

**Ventajas:**

- No necesita features de los ítems (útil cuando no existen, o son de mala
  calidad).
- Captura señales que ningún feature explicaría fácilmente (ej. "a la
  gente a la que le gusta el cine de autor iraní también le gusta el jazz
  experimental" — una correlación real en el comportamiento, pero que
  nunca se te ocurriría poner como feature a mano).
- Genera **serendipia**: recomendaciones sorprendentes pero relevantes,
  algo que content-based (que solo mira "más de lo mismo") estructuralmente
  no puede ofrecer.

**Desventaja principal:** sufre de cold-start tanto de usuario como de
ítem — sin interacciones, no hay señal colectiva de la que aprender.

La técnica clásica (y la que vamos a implementar hoy) para collaborative
filtering es la **factorización de matrices**.

## 9. Factorización de Matrices

La idea es aproximar la matriz de utilidad $R$ (de $m$ usuarios × $n$
ítems) como el producto de dos matrices mucho más pequeñas:

$$R \approx U \cdot V^{T}$$

donde:

- $U$ es una matriz $m \times k$: cada fila $u_u$ es el **vector latente**
  del usuario $u$.
- $V$ es una matriz $n \times k$: cada fila $v_i$ es el **vector latente**
  del ítem $i$.
- $k$ (típicamente entre 20 y 300) es la **dimensión latente**: el número
  de "factores ocultos" que el modelo aprende para describir gustos y
  características (no corresponden a géneros concretos como "acción" o
  "comedia" — son direcciones abstractas que el modelo descubre solas a
  partir de los datos, aunque a veces terminan correlacionándose con
  conceptos interpretables).

El rating predicho para el par $(u, i)$ es simplemente el producto punto
de sus vectores latentes:

$$\hat{r}(u,i) = u_u \cdot v_i^{T} = \sum_{f=1}^{k} u_{uf} \, v_{if}$$

### Función objetivo

Se entrena minimizando el error cuadrático **solo sobre los pares
observados** (no tiene sentido, ni es posible, comparar contra celdas
vacías), más un término de **regularización L2** para evitar sobreajuste
(recuerda: hay pocos pares observados por usuario/ítem, así que sin
regularización el modelo memoriza en vez de generalizar):

$$\min_{U,V} \sum_{(u,i) \in \text{observados}} \left( r_{ui} - u_u \cdot v_i^{T} \right)^2 \; + \; \lambda \left( \lVert U \rVert^2 + \lVert V \rVert^2 \right)$$

### ¿Cómo se optimiza esto?

Hay dos familias clásicas de optimización:

- **ALS (Alternating Least Squares):** se fija $V$ y se resuelve $U$
  exactamente por mínimos cuadrados (es un problema convexo en $U$ si $V$
  es constante); luego se fija $U$ y se resuelve $V$; se repite hasta
  converger. Cada paso es un problema de mínimos cuadrados cerrado, lo cual
  **paraleliza muy bien** (es la razón por la que Spark MLlib usa ALS como
  su algoritmo de recomendación por defecto — se puede repartir por filas o
  columnas entre nodos).
- **SGD (Stochastic Gradient Descent):** se recorren los pares observados
  (en orden aleatorio) y, para cada uno, se da un pequeño paso de gradiente
  que ajusta *simultáneamente* $u_u$ y $v_i$ para reducir el error de ese
  par puntual.

Ambos métodos llegan a soluciones de calidad similar. Hoy vamos a
implementar **SGD en numpy puro**: es bastante más simple de programar
desde cero que ALS (no hay que resolver sistemas de ecuaciones lineales en
cada paso) y es igual de ilustrativo para entender qué está aprendiendo el
modelo.

### Split de entrenamiento/evaluación

A diferencia de series de tiempo (Sesión 4), aquí **no hay fuga temporal**
que cuidar: cada fila de `ratings` es un par $(u,i)$ independiente, así que
un split aleatorio de filas (ocultando un subconjunto de celdas observadas
de $R$) es perfectamente válido. Usamos `random_state=42` para el split de
esta demo de clase — el assignment de esta sesión usa una semilla distinta
(`random_state=123`) sobre el mismo dataset, así que sus pares de
entrenamiento/evaluación no son los mismos que resolvemos aquí.

In [ ]:
train_df, test_df = train_test_split(ratings, test_size=0.2, random_state=RANDOM_STATE)
print("Train:", train_df.shape, " Test:", test_df.shape)

n_users_total = ratings["user_id"].max()
n_items_total = ratings["item_id"].max()
print("IDs de usuario hasta:", n_users_total, " IDs de película hasta:", n_items_total)

### Implementación de SGD para factorización de matrices, en numpy puro

Nota de rendimiento: entrenamos con épocas y $k$ modestos (documentados
abajo) para que el notebook completo corra en un par de minutos. Un sistema
de producción entrenaría muchas más épocas, sobre muchísimos más datos, y
probablemente con $k$ más grande — aquí priorizamos que el ejercicio sea
rápido de ejecutar y de entender.

In [ ]:
def factorizar_matriz_sgd(train_df: pd.DataFrame, n_users: int, n_items: int,
                           k: int = 20, lr: float = 0.01, reg: float = 0.05,
                           epochs: int = 20, seed: int = 42):
    '''Factorización de matrices R ≈ U @ V.T entrenada con SGD puro en numpy.

    Parameters
    ----------
    train_df : DataFrame con columnas user_id, item_id, rating (1-indexados).
    n_users, n_items : máximo user_id / item_id observado en todo el dataset
        (para dimensionar U y V; +1 porque los ids son 1-indexados).
    k : dimensión de los vectores latentes.
    lr : learning rate.
    reg : coeficiente de regularización L2 (lambda).
    epochs : número de pasadas completas sobre train_df.

    Returns
    -------
    U, V : matrices de factores latentes.
    history : RMSE de train al final de cada época (para ver convergencia).
    '''
    rng = np.random.default_rng(seed)
    U = rng.normal(loc=0.0, scale=0.1, size=(n_users + 1, k))
    V = rng.normal(loc=0.0, scale=0.1, size=(n_items + 1, k))

    users = train_df["user_id"].to_numpy()
    items = train_df["item_id"].to_numpy()
    obs_ratings = train_df["rating"].to_numpy(dtype=float)
    n_obs = len(users)
    order = np.arange(n_obs)

    history = []
    for epoch in range(epochs):
        rng.shuffle(order)
        sq_error_sum = 0.0
        for idx in order:
            u, i, r = users[idx], items[idx], obs_ratings[idx]
            pred = U[u] @ V[i]
            error = r - pred

            u_vec = U[u].copy()  # copia porque V[i] se actualiza con el valor viejo de U[u]
            U[u] += lr * (error * V[i] - reg * U[u])
            V[i] += lr * (error * u_vec - reg * V[i])

            sq_error_sum += error ** 2

        history.append(np.sqrt(sq_error_sum / n_obs))

    return U, V, history

In [ ]:
# Hiperparámetros elegidos para esta demo:
# - k=20: extremo bajo del rango típico (20-300), suficiente para un catálogo
#   de solo 1682 películas y rápido de entrenar.
# - epochs=20: con este dataset (80,000 pares de entrenamiento) 20 épocas
#   entrenan en unos ~15-20 segundos y ya alcanzan un buen punto antes de
#   empezar a sobreajustar fuerte (lo vemos con la brecha train/test más abajo).
# - reg=0.05: regularización L2 moderada, importante por la sparsity del problema.
U, V, history = factorizar_matriz_sgd(
    train_df, n_users_total, n_items_total, k=20, lr=0.01, reg=0.05, epochs=20, seed=RANDOM_STATE,
)

plt.figure(figsize=(6, 4))
plt.plot(range(1, len(history) + 1), history, marker="o")
plt.xlabel("Época")
plt.ylabel("RMSE de entrenamiento")
plt.title("Convergencia de la factorización con SGD")
plt.grid(alpha=0.3)
plt.show()

## 10. Métricas de evaluación — predicción de rating

- **RMSE** (Root Mean Squared Error):

$$\mathrm{RMSE} = \sqrt{\frac{1}{|T|}\sum_{(u,i) \in T} \left(r_{ui} - \hat{r}(u,i)\right)^2}$$

  Al elevar al cuadrado, **penaliza mucho más los errores grandes** que los
  pequeños — es sensible a outliers. Fue la métrica oficial del **Netflix
  Prize**.

- **MAE** (Mean Absolute Error):

$$\mathrm{MAE} = \frac{1}{|T|}\sum_{(u,i) \in T} \left| r_{ui} - \hat{r}(u,i)\right|$$

  Penaliza **proporcionalmente** al tamaño del error — más robusta a
  outliers que RMSE (un par de predicciones muy malas no domina la métrica
  como sí ocurre con RMSE).

In [ ]:
def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def mae(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.mean(np.abs(y_true - y_pred)))


test_users = test_df["user_id"].to_numpy()
test_items = test_df["item_id"].to_numpy()
test_ratings = test_df["rating"].to_numpy(dtype=float)

test_preds = np.array([U[u] @ V[i] for u, i in zip(test_users, test_items)])

print(f"RMSE en test: {rmse(test_ratings, test_preds):.4f}")
print(f"MAE  en test: {mae(test_ratings, test_preds):.4f}")
print(f"(para referencia, un baseline ingenuo que siempre predice el "
      f"promedio global de train da RMSE={rmse(test_ratings, np.full_like(test_ratings, train_df['rating'].mean())):.4f})")

Nuestro modelo de factorización mejora claramente el baseline "predecir
siempre el promedio". No es un modelo de nivel Netflix Prize (esos ganaron
con ensambles de decenas de modelos y muchísimo más cómputo), pero con ~50
líneas de numpy puro y un par de minutos de entrenamiento ya capturamos
buena parte de la señal en los datos.

## 11. Métricas de evaluación — ranking

Predecir bien el rating es útil, pero lo que casi siempre importa en
producción es el **orden**: ¿los primeros $K$ ítems que le mostramos al
usuario son los correctos? Estas métricas evalúan justamente eso.

- **Precision@K:** de los $K$ ítems que recomendamos, ¿qué fracción son
  relevantes?

$$\mathrm{Precision@K} = \frac{\#\ \text{ítems relevantes en el Top-K}}{K}$$

- **Recall@K:** de todos los ítems relevantes que existen para ese
  usuario, ¿qué fracción capturamos en el Top-K?

$$\mathrm{Recall@K} = \frac{\#\ \text{ítems relevantes en el Top-K}}{\#\ \text{ítems relevantes totales}}$$

- **MAP@K (Mean Average Precision):** el promedio del *Average Precision*
  (AP) de todos los usuarios. El AP de un usuario promedia la precisión
  medida en cada posición donde aparece un ítem relevante (le da más peso
  a acertar temprano en la lista, no solo a acertar en algún lugar):

$$\mathrm{AP@K} = \frac{1}{\min(K, \#\text{relevantes})}\sum_{k=1}^{K} P(k)\cdot \mathrm{rel}(k)$$

  donde $P(k)$ es la precisión en la posición $k$ y $\mathrm{rel}(k)$ es 1
  si el ítem en la posición $k$ es relevante (0 si no). $\mathrm{MAP@K}$ es
  el promedio de $\mathrm{AP@K}$ sobre todos los usuarios evaluados.

- **NDCG@K (Normalized Discounted Cumulative Gain):** descuenta la
  relevancia de un ítem **logarítmicamente según su posición** — un ítem
  relevante en la posición 1 vale mucho más que uno en la posición 10:

$$\mathrm{DCG@K} = \sum_{i=1}^{K} \frac{\mathrm{rel}_i}{\log_2(i+1)}$$

  Se normaliza dividiendo por el **DCG ideal** ($\mathrm{IDCG@K}$): el DCG
  que se obtendría si los ítems estuvieran ordenados exactamente por su
  relevancia real, de mayor a menor:

$$\mathrm{NDCG@K} = \frac{\mathrm{DCG@K}}{\mathrm{IDCG@K}}$$

  $\mathrm{NDCG@K} \in [0, 1]$, donde 1 significa que el orden que dimos es
  el orden ideal posible.

In [ ]:
def precision_at_k(recomendados: list, relevantes: set, k: int) -> float:
    top_k = recomendados[:k]
    hits = sum(1 for item in top_k if item in relevantes)
    return hits / k


def recall_at_k(recomendados: list, relevantes: set, k: int) -> float:
    if len(relevantes) == 0:
        return 0.0
    top_k = recomendados[:k]
    hits = sum(1 for item in top_k if item in relevantes)
    return hits / len(relevantes)


def average_precision_at_k(recomendados: list, relevantes: set, k: int) -> float:
    if len(relevantes) == 0:
        return 0.0
    top_k = recomendados[:k]
    hits = 0
    suma_precisiones = 0.0
    for pos, item in enumerate(top_k, start=1):
        if item in relevantes:
            hits += 1
            suma_precisiones += hits / pos
    return suma_precisiones / min(k, len(relevantes))


def dcg_at_k(relevancias: list, k: int) -> float:
    '''relevancias[i] = relevancia (graduada) del ítem en la posición i+1.'''
    dcg = 0.0
    for pos, rel in enumerate(relevancias[:k], start=1):
        dcg += rel / np.log2(pos + 1)
    return dcg


def ndcg_at_k(recomendados: list, relevancia_real: dict, k: int) -> float:
    '''relevancia_real: dict item_id -> relevancia graduada (ej. el rating real).'''
    relevancias_obtenidas = [relevancia_real.get(item, 0) for item in recomendados[:k]]
    dcg = dcg_at_k(relevancias_obtenidas, k)

    relevancias_ideales = sorted(relevancia_real.values(), reverse=True)
    idcg = dcg_at_k(relevancias_ideales, k)

    return dcg / idcg if idcg > 0 else 0.0

### Generando recomendaciones a partir del modelo de factorización

Para evaluar ranking necesitamos, para cada usuario, una lista **ordenada**
de recomendaciones sobre ítems que *no* vio en train (el modelo no debería
"recomendar" lo que el usuario ya calificó). Como el catálogo es chico
(1682 películas), podemos puntuar todo el catálogo de una vez con
`U @ V.T` — en un sistema real de millones de ítems esto es justo lo que
la etapa de *retrieval* evita tener que hacer.

Usamos como "verdad" (ground truth) los ítems que cada usuario calificó en
el **test set** — es una limitación conocida de la evaluación offline:
solo sabemos si al usuario "le gustó" lo que efectivamente calificó, no
tenemos ground truth sobre el resto del catálogo (que simplemente no vio,
no necesariamente porque no le fuera a gustar).

In [ ]:
pred_matrix = U @ V.T  # (n_users+1) x (n_items+1), puntúa TODO el catálogo

# Máscara: no recomendar ítems que el usuario ya vio en TRAIN
vistos_en_train = train_df.groupby("user_id")["item_id"].apply(set).to_dict()


def top_k_recomendaciones(user_id: int, k: int = 10) -> list:
    scores = pred_matrix[user_id].copy()
    scores[0] = -np.inf  # item_id 0 no existe
    for item_id in vistos_en_train.get(user_id, set()):
        scores[item_id] = -np.inf
    return list(np.argsort(-scores)[:k])


# Relevancia por usuario a partir del TEST set (lo que el usuario calificó y no vimos en train)
relevancia_por_usuario = test_df.groupby("user_id").apply(
    lambda g: dict(zip(g["item_id"], g["rating"])), include_groups=False
).to_dict()

UMBRAL_RELEVANTE = 4  # rating >= 4 se considera "relevante" para precision/recall/MAP (binario)

# Evaluamos sobre usuarios con al menos 10 ratings en test (para que las métricas de ranking
# tengan suficiente señal para ser informativas)
usuarios_para_evaluar = [u for u, d in relevancia_por_usuario.items() if len(d) >= 10][:15]
print(f"Evaluando ranking sobre {len(usuarios_para_evaluar)} usuarios de ejemplo")

K = 10
filas = []
for u in usuarios_para_evaluar:
    recomendados = top_k_recomendaciones(u, k=K)
    relevancia_graduada = relevancia_por_usuario[u]
    relevantes_binario = {item for item, r in relevancia_graduada.items() if r >= UMBRAL_RELEVANTE}

    filas.append({
        "user_id": u,
        "n_test": len(relevancia_graduada),
        "n_relevantes": len(relevantes_binario),
        f"Precision@{K}": precision_at_k(recomendados, relevantes_binario, K),
        f"Recall@{K}": recall_at_k(recomendados, relevantes_binario, K),
        f"AP@{K}": average_precision_at_k(recomendados, relevantes_binario, K),
        f"NDCG@{K}": ndcg_at_k(recomendados, relevancia_graduada, K),
    })

resultados_ranking = pd.DataFrame(filas)
print(f"\nMAP@{K} (promedio de AP@{K} sobre los usuarios evaluados): {resultados_ranking[f'AP@{K}'].mean():.4f}")
resultados_ranking

## 12. Más allá del accuracy

Un recomendador con excelente RMSE o NDCG puede, aun así, ser un mal
producto. Algunas dimensiones adicionales que los equipos de recomendación
en producción monitorean activamente:

- **Cobertura (coverage):** ¿qué fracción del catálogo el sistema *llega a
  recomendar* alguna vez? Un modelo puede tener excelente accuracy
  recomendando siempre las mismas 20 películas populares a todo el mundo
  — y aun así ser un mal recomendador, porque nunca le da visibilidad al
  resto del catálogo.
- **Diversidad:** ¿qué tan distintos son los ítems *dentro de una misma
  lista* recomendada a un usuario? 10 secuelas de la misma saga no es una
  lista diversa, aunque cada una individualmente sea "relevante".
- **Novedad:** ¿qué tan poco populares son, en promedio, los ítems que
  recomendamos? Recomendar siempre lo más popular (fácil de acertar, alto
  accuracy) no aporta descubrimiento — el usuario probablemente ya conocía
  esos ítems.
- **Serendipia:** recomendaciones que son inesperadas (el usuario no las
  hubiera encontrado por su cuenta) *pero* relevantes. Es la métrica más
  difícil de cuantificar y la que más distingue un buen sistema de
  collaborative filtering de uno content-based puro.
- **Justicia / fairness:** ¿el sistema da visibilidad equitativa a
  distintos productores/creadores/ítems, o sistemáticamente favorece a
  unos pocos (ej. por retroalimentación positiva: lo popular se recomienda
  más, se vuelve más popular, se recomienda aún más)?

Ninguna de estas reemplaza al accuracy — lo complementan. Un sistema real
casi siempre optimiza una combinación de varias de estas señales (a veces
explícitamente en la función objetivo, a veces en la etapa de
re-ranking).

In [ ]:
# Demo ligera: cobertura y novedad de las recomendaciones del modelo de factorización,
# para todos los usuarios del dataset.
K_DEMO = 10
popularidad = train_df.groupby("item_id").size()  # veces que cada película fue calificada en train
total_interacciones_train = popularidad.sum()

items_recomendados = set()
novedades_por_usuario = []

for u in range(1, n_users_total + 1):
    recs = top_k_recomendaciones(u, k=K_DEMO)
    items_recomendados.update(recs)

    pops = popularidad.reindex(recs).fillna(1)  # fillna por si acaso: nunca vista en train
    prob_pop = pops / total_interacciones_train
    novedad_lista = float(np.mean(-np.log2(prob_pop)))  # a menor popularidad, mayor "auto-información"
    novedades_por_usuario.append(novedad_lista)

cobertura = len(items_recomendados) / n_items_total
novedad_promedio = float(np.mean(novedades_por_usuario))

print(f"Cobertura del catálogo (Top-{K_DEMO} para todos los usuarios): {cobertura:.2%} "
      f"({len(items_recomendados)} de {n_items_total} películas)")
print(f"Novedad promedio de las recomendaciones (bits, más alto = menos popular): {novedad_promedio:.2f}")

Interpreta estos dos números con cautela: no hay un valor "bueno" o
"malo" absoluto — dependen del negocio. Una cobertura baja no es
necesariamente un problema si el objetivo del producto es maximizar
engagement inmediato; sí lo es si el objetivo incluye descubrimiento o
justicia entre productores. Lo importante es que estos números **existen**
y se pueden monitorear junto con RMSE/NDCG, no en su reemplazo.

## 13. Más allá de la factorización de matrices clásica

La factorización de matrices que implementamos hoy es potente y simple,
pero tiene limitaciones estructurales:

- El producto interno $u_u \cdot v_i^T$ es **lineal** — no puede capturar
  interacciones complejas entre factores latentes (ej. "a este usuario le
  gusta A y B juntos, pero no A o B por separado").
- **Ignora el contexto**: hora del día, dispositivo, ubicación, qué vio el
  usuario hace 5 minutos. Un mismo usuario puede querer recomendaciones
  distintas viendo TV el sábado en la noche vs. en el bus un martes en la
  mañana — el MF clásico no distingue esto.
- **No incorpora atributos ricos** de texto o imagen de forma nativa (más
  allá de features tabulares simples como género).

Estas limitaciones motivan a los modelos híbridos y de *deep learning*
para recomendación, que solo mencionamos conceptualmente (no los
implementamos hoy):

- **Wide & Deep (Google, 2016):** combina un componente "wide" (lineal,
  con features cruzados, bueno para memorizar patrones específicos) con
  uno "deep" (una red neuronal, buena para generalizar a combinaciones no
  vistas de features) entrenados conjuntamente.
- **Two-tower neural networks:** una red aprende un embedding del usuario
  (con su contexto) y otra un embedding del ítem, de forma independiente
  (dos "torres"), entrenadas para que el producto punto de ambos
  embeddings se acerque a la afinidad real. Es, en esencia, una
  factorización de matrices "no lineal" — cada torre puede incorporar
  features ricos (texto, contexto, historial reciente) antes de producir
  el embedding final. Es la arquitectura estándar hoy en la etapa de
  **retrieval** que vimos en la sección 4, precisamente porque los
  embeddings de ítem se pueden precalcular e indexar con ANN (FAISS/HNSW)
  para retrieval en milisegundos sobre catálogos enormes.

## Resumen de la sesión

- Un sistema de recomendación completa una **matriz de utilidad**
  extremadamente **sparse**, ya sea prediciendo un rating o resolviendo
  una tarea de **ranking**.
- El feedback puede ser **explícito** (limpio, escaso) o **implícito**
  (ruidoso, abundante) — la mayoría de sistemas reales viven del segundo.
- A gran escala, los sistemas usan un pipeline de **retrieval → ranking →
  re-ranking** para no tener que puntuar todo el catálogo con un modelo
  pesado.
- **Content-based filtering** usa atributos del ítem + similitud coseno;
  resuelve bien el cold-start de ítems y es explicable, pero con poca
  serendipia.
- **Collaborative filtering** (aquí, vía **factorización de matrices**
  $R \approx U V^T$, entrenada con SGD) aprende patrones puramente
  colectivos; más serendipia, pero sufre cold-start de usuario e ítem.
- Evaluamos con métricas de regresión (**RMSE**, **MAE**) y de ranking
  (**Precision@K**, **Recall@K**, **MAP@K**, **NDCG@K**), todas
  implementadas a mano — y discutimos que el accuracy no basta:
  **cobertura, diversidad, novedad, serendipia y fairness** también
  importan.
- Los modelos modernos (Wide & Deep, two-tower) extienden estas ideas con
  no linealidad y contexto, pero el esqueleto conceptual —completar una
  matriz de utilidad sparse, evaluando con las métricas correctas— es
  exactamente el mismo que construimos hoy desde cero.

### Próxima sesión

En la Sesión 5 cerramos el curso con **aprendizaje no supervisado**:
reducción de dimensionalidad (PCA, y UMAP a nivel conceptual) y clustering
(jerárquico, K-Means, DBSCAN). Es el otro gran bloque del mapa que abrimos
en la Sesión 1, cuando separamos supervisado de no supervisado — y la
sparsity y las nociones de similitud que usamos hoy reaparecen allí.

Este es también un buen momento para revisar el reto de esta sesión
(`assignments/recomendador-peliculas/`) y, si el tiempo lo permite, volver
sobre los retos anteriores que hayan quedado pendientes.

---

# Ejercicio

Usan lo construido arriba: `ratings`, `movies`, `train_df` / `test_df`, `factorizar_matriz_sgd` y las funciones de recomendación y de métricas.

| # | Parte | Tiempo sugerido |
|---|---|---|
| 1 | Ajustar el número de factores latentes | 10 min |
| 2 | Recomendaciones y cold-start | 10 min |
| 3 | Métricas de ranking | 10 min |

Si una parte se atasca, pasen a la siguiente: valen más las tres intentadas que una perfecta.

### Ejercicio 1

Entrena la factorización de matrices con al menos **tres valores distintos
de $k$** (ej. 5, 20, 50) manteniendo `lr`, `reg` y `epochs` fijos, y
compara el **RMSE en test** de cada uno. ¿$k$ más grande siempre da mejor
resultado en este dataset? ¿Por qué podría no ser así, dado lo que vimos
sobre sparsity?

In [ ]:
# TODO: entrena factorizar_matriz_sgd con k=5, k=20 y k=50 (mismos train_df, lr, reg, epochs)
# y reporta el RMSE de test de cada uno en una tabla.

### Ejercicio 2

Elige un usuario (puede ser el `usuario_demo` de la sección de
content-based, o cualquier otro con varias interacciones) y compara sus
**top-10 recomendaciones de content-based** contra sus **top-10
recomendaciones del modelo de factorización (collaborative filtering)**.
¿Qué tan distintas son las dos listas? ¿A qué atribuyes las diferencias?

In [ ]:
# TODO: usa recomendar_content_based(...) y top_k_recomendaciones(...) para el mismo user_id
# y compara las dos listas de títulos (puedes usar movies para mapear item_id -> title).

### Ejercicio 3

La muestra de 15 usuarios de `resultados_ranking` es demasiado chica para
sacar conclusiones robustas. Calcula `NDCG@10` para **todos** los usuarios
que tengan al menos 5 ratings en `test_df` (unos 800), y para cada uno
guarda también cuántas interacciones tiene en `train_df`. Con esa tabla:
¿hay correlación entre pocas interacciones de entrenamiento y mal
desempeño de ranking? Grafica `n_train` vs. `NDCG@10` (scatter) y calcula
la correlación. Relaciónalo con lo que vimos sobre cold-start.

In [ ]:
# TODO: para cada usuario en relevancia_por_usuario con len(...) >= 5, calcula:
#   - top_k_recomendaciones(user_id, k=10)
#   - ndcg_at_k(...) contra su relevancia (graduada) de test
#   - cuántas filas tiene en train_df
# junta todo en un DataFrame y calcula la correlación entre n_train y NDCG@10.